# Using APIs Through the Lens of PARADISEC

***
## Introduction

If you were to begin programming a website or app for something, anything, odds are you'll eventually want to interact with something beyond your own software. You wouldn't want to have to navigate the backend of a large piece of external software in hopes of learning how to interact with it, and that's assuming you know of a way to interact with that backend in the first place. These kinds of problems are what an *Application Programming Interface*, or [**API**](https://www.geeksforgeeks.org/websites-apps/how-to-use-an-api-the-complete-guide/), looks to solve. An **API** is a set of rules and protocols that allow one software application to interact with another, essentially acting as a middleman that facilitates communication between two separate applications. This removes the burden from external developers needing to create their own methods of interaction and allows developers to ensure that their application is being interacted with in a desired and efficient manner.

There are many different types of **API**s, with the most common being *Representational State Transfer*, or **REST**. This uses HTTP requests to `GET`, `POST`, `PUT`, and `DELETE` data, and is both simple and scalable, hence its wide usage. In this notebook we'll be looking at a partially similar architecture known as [**GraphQL**](https://graphql.org/learn/).  This is a query language that allows clients to request exactly the data they need, preventing under/over fetching. We'll be working with this type of **API** as it is the type used by [**PARADISEC**](https://www.paradisec.org.au/), the archive whose **API** we'll be using as an example.

**Why GraphQL?**

It can help to think of a **GraphQL** *schema* less like a list of **API** endpoints and more like a *knowledge graph*: every *type* is a node, and every *field* that points to another *type* is an edge connecting it to a related node. A query is really just a description of a path (or several paths) through that graph - starting at a root node and specifying exactly which nodes and *field*s to follow and return.

This graph-shaped design gives **GraphQL** a few key advantages over more traditional **API** architectures like **REST**:

- **Precise data fetching.** Clients ask for exactly the *field*s they need, no more and no less - avoiding both *over-fetching* (getting back extra data you'll just discard) and *under-fetching* (not getting enough, and needing a follow-up request to fill in the gaps).
- **Related data in one request.** Because the *schema* is a graph of connected *type*s, a single query can traverse relationships - for example, an item and its associated university - that would otherwise require several round-trips to separate **REST** endpoints.
- **One endpoint for everything.** Rather than exposing a different URL for every resource (`/items`, `/collections`, `/universities`, etc.), a **GraphQL** **API** typically exposes just one *endpoint*, with the *schema* determining what can be queried through it.
- **A self-describing schema.** Every *type*, *field*, and *argument* is declared up front, so the *schema* can be explored and validated automatically (as we'll do with **PARADISEC**'s *schema* shortly) - useful even when written *documentation* is limited.

Keep this graph analogy in mind as we walk through **GraphQL**'s *type*s and *field*s below - it's exactly why the "graph" is in **GraphQL**.

***
## PARADISEC

*The Pacific And Regional Archive for Digital Sources in Endangered Cultures*, or **PARADISEC**, is a digital archive of records of many small cultures and languages whose primary goal is to safely preserve that would have otherwise been lost. We will explore a bit of their collection while making use of their **GraphQL API** in order to demonstrate both the uses of **API**s and some of the problems one might run into.

We should first understand a bit of how the archive of **PARADISEC** actually works. The archive is organized into *collections*, which themselves are organized around a single theme, such as a single trip of a collector. Every *collection* contains some amount of *items*, which refer to a single document, event or recording. Every item has some amount of *essences*, which are the actual individual files such as `.pdf`s, `.wav`s or `.jpeg`s. There is additional metadata associated with each of these - that metadata is what we will be accessing through their **API**.

<h3><center>Collection</center>
<h4><center>&darr;</center>
<h4><center>Item</center>
<h4><center>&darr;</center>
<h5><center>Essence</center>

***
## GraphQL

We also must first understand how a **GraphQL** **API** works. This will not be an in-depth explanation of all of the features of such an API, but rather a crash course in the basics that we will be working with.

**GraphQL** organizes data using *types* and *fields*:

- A ***type*** defines the structure of a piece of data.
- A ***field*** is a specific piece of data within a *type*. Every *field* is either an *Object* or a *Scalar*.
  - An ***Object*** is a *type* defined by the developers of the **API**, in whatever way makes sense for their data, and can contain any number of *field*s.
  - A ***Scalar*** represents a leaf value of a query - it has no sub-fields, so it's the end result of a query path.

For example, recall the `Collection` &rarr; `Item` &rarr; `Essence` hierarchy from earlier: `Item` is one of **PARADISEC**'s *Object* *type*s, representing a single document, event, or recording. `Item` has *field*s like `full_identifier` and `title`, both of which are `String` *Scalar*s - we'll see these *field*s in action in the example query below.

The default *Scalar*s are `Int`, `Float`, `String`, `Boolean`, and `ID` (a unique identifier serialized the same way as a string, but not intended to be human-readable).

**API**s can also define custom *Scalar* *type*s. **PARADISEC** has two:

- `BigInt`: formatted as an integer, but may be larger than 32 bits, so it's encoded as a string instead.
- `ISO8601DateTime`: follows an international standard for representing dates and times.

This is a lot of information, so let's walk through an example of what a query actually looks like and identify each of the parts. The example **PARADISEC** themselves provide as a basic query looks like this:

```json
{
  items(full_identifier: "ABC") {
    total
    next_page
    results {
      full_identifier
      title
    }
  }
}
```

The first thing to point out is the curly brackets:

- Every time you move down through the hierarchy, you surround the next level in curly brackets.
- The whole query is surrounded by curly brackets too, because everything is actually contained within a `query` *type*. This is known as the root object, and is the entry point for communicating with the **API**.
- Since `query` is the default, we can skip straight to the curly brackets: the first line `{` means the same thing as `query {`. We'll come back to this idea later.

Within the root object of `query`, there are a few different *field*s you can query on, as defined by **PARADISEC** - one of them being `items`, as we see in the example above. This time, though, `items` is followed by parentheses containing additional text before the next set of curly brackets opens. That's because every *field* in **GraphQL** can have zero or more named *argument*s:

- *Argument*s can be required or optional; if optional, a default value can also be assigned.
- In this example, `items` has many optional *argument*s, one of which is named `full_identifier` and accepts a `String`.
- `items(full_identifier: "ABC")` says we want to query on `items`, supplying `"ABC"` as the `full_identifier` *argument*, while all other optional *argument*s are left at their default value.

Anytime *argument*s are passed, they follow the same format - the *argument* name, then `:`, then the desired value:<br>`type_name(first_argument_name: first_argument_value, second_argument_name: second_argument_value, ...)`

By querying on `items`, we get back an `ItemResult`, which has three *field*s:

- `total`, an `Int`
- `next_page`, an `Int`
- `results`, a list of `Item`s

We query on all three in the example above. `Int` is one of the *Scalar*s that function as leaf values, which is why `total` and `next_page` aren't followed by curly brackets. `Item`, on the other hand, is an *Object*, so `results` is followed by more curly brackets and more sub-fields to query on. Note that this is required - you have to end on leaf values, so simply ending with `results` and no curly brackets won't return a complete `Item` (or anything else); it will instead return an error.

`Item` has many *field*s, two of which appear above:

- `full_identifier`, a `String`
- `title`, a `String`

Again, `String` is a *Scalar*, so neither is followed by more curly brackets. After closing all the open curly brackets, we've reached the end of the query!

At this point we should be beginning to understand the basic structure of a **GraphQL** query. However, what do we actually get back from it? **PARADISEC** describes the above example thusly: "This query finds all items which have 'ABC' in their identifier (including collection identifier), and then lists the full identifier and title of results." So, let's see what we would expect the output to look like:

```json
{
  "data": {
    "items": {
      "total": 1,
      "next_page": null,
      "results": [
        {
          "full_identifier": "WD1-ABC",
          "title": "Tum-why-village-moved-here"
        }
      ]
    }
  }
}
```

While **GraphQL** queries are not [JSON](https://www.json.org/json-en.html), both the actual request sent to the server and the response recieved from it are typically (though not always!) formatted as JSON. We can see that the above output is JSON that still closely resembles the structure of our original query.

The first thing to notice is that all the information we recived is wrapped in `"data"`; in any **GraphQL** request, the result of the execution of the requested operation is always wrapped in `"data"`, and information about any raised errors is wrapped in `"errors"`.

Moving down to the next level in the hierarchy, we see the structure of our original query surfacing. We first run into `"items"`; we then move down another level in the hierarchy and see its sub-fields we called on. 

- `"total"` returned a result of `1`, which matches our expected return type of `Int`
- `"next_page"` returned a result of `null`. This is because *type*s in **GraphQL** are assumed to be *nullable*, meaning they may not actually have a value and could instead return `null` (this will be expanded upon a bit further on).
- `"results"` looks slightly different from the others: instead of being immediately followed by curly brackets `{ }` and then the contained sub-fields, it is first followed by square `[ ]`  brackets. This is because, if we think back to the expected returns of each *field*, `results` would return a <u>list</u> of `Item`s, not just `Item`. The returned list is contained within normal brackets, and then each member of that list is surrounded by its own set of curly brackets. With that made clear, looking at `"full_identifier"` and `"title"` for the one `Item` we got back, these each returned a `String`, again matching our expected return *type*s.

With that, we should be developing a basic understanding of **GraphQL** queries and their responses, and can start exploring **PARADISEC** further to cement our understanding.

***
## Initial Exploration

Understanding the structure is great, but how did we know what to look for? How could you know what custom *Scalar*s **PARADISEC** defined, or any of **PARADISEC**'s custom queries and *type*s and *field*s, or what passing in certain *argument*s will do? Well, there are essentially two slightly different answers to those questions. In terms of knowing how **PARADISEC**'s specific **GraphQL** **API** is structured, that information was gleaned through their *schema* definition. A **GraphQL** *schema* is essentially a blueprint for the **API** that defines all the various queries, *type*s, *field*s, etc. [Take a look!](https://admin-catalog.paradisec.org.au/paradisec.graphql)

### The Schema is Alphabetical
A couple things worth pointing out here, the first being that the schema is organized alphabetically, not by function or grouped by related things. Additionally, you'll see some text surrounded by triple double quotes, `"""`, these are comments meant for humans reading the *schema*.

### Special Characters in Return Types
You may also have noticed that the return *type*s for some fields look a bit different from the rest, with some being surrounded in brackets and others being followed by an exclamation mark. 

- The brackets indicate that the return is a list of some *type*, rather than a single *type*. 
- The exclamation mark signifies that the return is <u>not</u> *nullable* (the assumption we just learned about), and if queried on will always return something. The exclamation mark is also used when defining *argument*s to differentiate between those that are optional and those that are required; *argument*s that take in a *type* followed by an exclamation mark are required and do not accept null values. 
- The exclamation mark and brackets can also be combined: an exclamation mark within the brackets signifies that the list cannot have null members, an exclamation mark outside the brackets signifies that the list itself cannot be null, and exclamation marks both within and outside the brackets mean both that the list cannot be null <u>and</u> cannot have any null members (note that in all combinations, an empty list is valid).

### The 'Query' Type:  The *Root* of the Graph
All of the *field*s within the `Query` *type* are those *field*s that we can access by making a query request, such as the `items` *field* that we saw before. Again, by examining the *schema* you can see how we knew the *argument*s in `items`, what *type* they took in, and what the return *type* was, and we can determine how to use any of the other query *field*s and what information any *type* has in the same way.

### Documentation Discrepancies
You may notice something odd when looking more into the various *field*s within the `Query` *type* - the first two *field*s, `collection` and `essence`, have identical comments above them: `Find a collection by identifier. e.g. NT1`. This makes sense for a *field* named `collection`, but doesn't seem to fit for a *field* named `essence`, does it? This divide becomes increasingly clear when looking at the *argument*s: `collection` takes in `identifier`, but `essence` takes in both `fullIdentifier` and `filename`. 

This is where we find one of the problems you may encounter when working with **API**s: forms of *documentation* can sometimes be outdated or inaccurate somehow. In this specific instance, we can assume that the comment for `collection` was copied over and then intended to be edited to match `essence`, but was forgotten about. Whatever the actual reason, sometimes we run into cases like this where explanation for some part of an **API** doesn't match what it's supposedly describing. 

In instances like this, trust your instinct, but don't just accept confusion! If you see something that doesn't make sense, try testing it by itself to determine if the confusion stems from you or truly from a problem with the **API** *documentation*, and determine what the actual function of that thing is.

### All the Fields on an `Item`

So far we've only queried `full_identifier` and `title`, which is fairly rudimentary. Real research questions usually need much more - which languages are involved, which collection an item belongs to, which university holds it, links to the actual recordings, and so on. Since the *schema* fully defines the `Item` *type*, we can see every *field* available to us, organized here by whether each one is a *Scalar* (a leaf value) or an *Object* / list (which needs its own sub-*field*s).

**Scalar fields** (query these directly, no sub-fields needed):

| Field | Type |
|---|---|
| `access_class` | `String` |
| `access_condition_name` | `String` |
| `access_narrative` | `String` |
| `born_digital` | `Boolean` |
| `citation` | `String` |
| `created_at` | `ISO8601DateTime` |
| `description` | `String` |
| `dialect` | `String` |
| `digitised_on` | `String` |
| `doi` | `String` |
| `doi_json` | `String` |
| `essences_count` | `Int` |
| `full_identifier` | `String!` |
| `id` | `ID!` |
| `identifier` | `String!` |
| `ingest_notes` | `String` |
| `language` | `String` |
| `metadata_exportable` | `Boolean!` |
| `original_media` | `String` |
| `originated_on` | `String` |
| `originated_on_narrative` | `String` |
| `permalink` | `String!` |
| `private` | `Boolean` |
| `public` | `Boolean` |
| `received_on` | `String` |
| `region` | `String` |
| `title` | `String` |
| `tracking` | `String` |
| `updated_at` | `ISO8601DateTime` |

**Object fields** (a single related *Object* - needs its own `{ }` block with sub-*field*s):

| Field | Type |
|---|---|
| `access_condition` | `AccessCondition` |
| `boundaries` | `Boundary` |
| `collection` | `Collection!` |
| `collector` | `Person!` |
| `discourse_type` | `DiscourseType` |
| `operator` | `Person` |
| `university` | `University` |

**List fields** (a list of related *Object*s - also needs its own `{ }` block with sub-*field*s):

| Field | Type |
|---|---|
| `content_languages` | `[Language]` |
| `countries` | `[Country]` |
| `data_categories` | `[DataCategory]` |
| `data_types` | `[DataType]` |
| `essences` | `[Essence]` |
| `item_agents` | `[Agent]` |
| `subject_languages` | `[Language]` |

Putting this together, we can write a much richer query than our original example:

```json
{
  items(full_identifier: "AA1-001") {
    results {
      full_identifier
      title
      description
      region
      collection {
        title
      }
      collector {
        name
      }
      university {
        name
      }
      essences {
        filename
        permalink
      }
      content_languages {
        name
      }
    }
  }
}
```

None of this field-by-field detail is written up anywhere in prose - it only exists by reading the *schema* itself, which is exactly the kind of "documentation via schema" we discussed above.

***
## Accessing an API

Now we've looked through **PARADISEC**'s *schema* and should be ready to start making our own queries. In this notebook, we'll be using [Python](https://www.python.org/) to do so. However, we once again have to take a step back - how do we actually communicate with an **API** in the first place?

No matter the kind, every **API** has at least one *endpoint*. In this context, an *endpoint* is simply the URL where an **API** can be accessed. It's possible to have multiple *endpoint*s that expose different resources/functions. For a **GraphQL** **API**, because the different functions are covered by the various *field*s that fall under `Query`, there is typically only one *endpoint*. Let's store the *endpoint* for **PARADISEC**'s **GraphQL** **API** below.

In [1]:
API_URL = 'https://admin-catalog.paradisec.org.au/graphql'

To actually interact with web resources, we need the ability to send HTTP requests. There's a python library that makes this process fairly simple, called [Requests](https://docs.python-requests.org/en/latest/index.html); let's import that library below.

In [ ]:
import requests
import getpass


### Making a Request
With both the *endpoint* and a library to help us make HTTP requests, we should be good to go, right? Let's use the example query for now. Typically, if we just want to retrieve information back, we would use `requests.get()`. However, we'll be using `requests.post()` instead. This is because while `POST` requests are generally used to create or change data, they are also used to trigger actions such as a login, as `POST` requests include a request body in which additional data can be sent. When communicating with a **GraphQL** **API**, we need to send our query along with our request, which can be done via a request body, hence the usage of a `POST` request.

### Returning the Data to JSON
In order to help maintain readability, we'll store our query in a variable first as a block string. We'll then call `requests.post()` passing both our *endpoint* and our query. Our query will be passed into the `json` *argument* under the `"query"` designation, which will do the work of serializing our query into JSON and telling the **API** that it actually is a query. We'll store the result of the request in another variable.

In [3]:
query = '''
    {
        items(full_identifier: "ABC") {
            total
            next_page
            results {
                full_identifier
                title
            }
        }
    }
'''
response = requests.post(API_URL, json={'query': query})

(You may have noticed some strings being denoted by `''` and others by `""`. In Python, both `''` and `""` can denote strings, but `''` is the more common convention, so that's what we're using in this notebook. However, JSON only denotes strings as `""`, so when writing our queries, we must wrap any passed string arguments in `""`.)

This is the general format that every request we send to the **API** will follow. Most requests made with Requests will return a `Response` object, which contains the server's response and has a variety of methods that can access specific parts of the response. Again, because **GraphQL** requests and responses are generally formatted as JSON, we'll want to use the `.json()` method to get back the main information we care about. Since we used the example query, it should match the expected output from above. Let's run the code below and see!

In [4]:
response.json()

{'errors': [{'message': 'Must be logged in to query Nabu'}]}

...huh, not what we expected. We got an error back instead of any data. Although, if we look at the error message, it's pretty obvious what went wrong - we need to be logged in!

Many **API**s require some form of [*authentication*](https://www.geeksforgeeks.org/ethical-hacking/what-is-api-authentication-definition-and-working/). This ensures that only authorized clients can interact with the **API** and its data. There's a variety of methods through which **API**s deal with *authentication*. For example, free **API**s often use *Basic Authentication*, in which a username and password are sent along in a header of the request. In fact, Requests has an `auth` *argument* in `.get()` that takes in a username and password specifically for this method of *authentication*. Paid **API**s often use *Bearer Token Authentication*, in which a bearer token is obtained from an *authentication* server and is then included in the Authorization header of the request. These are just a few examples, and there's many more *authentication* methods out there.

**PARADISEC**'s **API** is a bit different. Their catalog has a login system that's required to view any items, and all requests sent to their **API** look for cookies from that login process on their catalog website, rather than any authentication through the actual **API**. Because of that, we're going to have to do a bit of extra coding before we start making requests. Before that, though, first [sign up](https://admin-catalog.paradisec.org.au/users/sign_up) on **PARADISEC** if you haven't done so before, then enter your log in information below.

In [ ]:
email = input('Email:')
password = getpass.getpass('Password:')

Now that we have login information, we can program that extra piece we need. The code is below, followed by a basic explanation of each part.

In [ ]:
from bs4 import BeautifulSoup

session = requests.Session()

login_page = session.get('https://admin-catalog.paradisec.org.au/users/sign_in')
soup = BeautifulSoup(login_page.text, 'html.parser')
csrf = soup.find('input', {'name': 'authenticity_token'})['value']

response = session.post(
    'https://admin-catalog.paradisec.org.au/users/sign_in',
    data={
        'authenticity_token': csrf,
        'user[email]': email,
        'user[password]': password,
    }
)

try:
    response.raise_for_status()
    print('Login Successful!')
except: print('Login Failed...')

[Beautiful Soup](https://beautiful-soup-4.readthedocs.io/en/latest/) is a library that makes it easy to parse through HTML and XML, so we're importing it to more easily parse the data within the login page. We also set up a [session](https://www.geeksforgeeks.org/python/session-objects-python-requests/) through Requests. A session allows us to persist certain parameters and cookies across multiple HTTP requests. Remember, the **API** is looking for a cookie that says we're logged in, so by making all our requests through this session, we only need to log in once and it'll pass that cookie along with all our future requests.

We first load the login page through our new session and store the HTML. We then use Beautiful Soup to turn that HTML into a structured object we can more easily search through. Finally, we retrieve a [**CSRF** token](https://brightsec.com/blog/csrf-token/) from where we would expect it to be.

Put simply, [**CSRF**](https://owasp.org/www-community/attacks/csrf) (short for *Cross-site Request Forgery*) is a form of cyber attack that forces a user to execute unwanted actions on a web application they're authenticated on. A **CSRF** token is a protection against this - a unique token is generated for each individual user session and is embedded in web forms or requests. When the user submits a form, the token is checked to ensure it matches what was given, confirming the action's legitimacy as attackers can't read or predict the token's value. That's why we can't just log into **PARADISEC** immediately, as we need to first load the webpage and retrieve the **CSRF** token to pass along with our email and password.

Now that we have everything we need, we can make another request to the login page, giving it all the needed login information. At this point, we should be logged in, but we added a check just in case! Every time a request is made, [status codes](https://www.geeksforgeeks.org/computer-networks/what-are-http-status-codes/) are returned by the server. Common status codes include: `200`, meaning the request was successful; `400`, meaning the request was invalid; `401`, meaning the request was unauthorized; and `404`, meaning the document at the specified URL doesn't exist. Requests' method `raise_for_status()` checks the status code; if it indicates a successful response, nothing happens, and if it indicates an error of some kind, an `HTTPError` is raised with details.

If you saw the message `Login Successful!`, then no error was raised from the login request, and you shoud be logged in and ready to move on! If you saw the message `Login Failed...`, then an error was raised and you're not logged in - ensure your login information is correct and try re-entering your information before attempting to log in again (if needed, you can move `response.raise_for_status()` outside of the try/except block to read the actual error information).

***
## Making a Query

Let's try running the example query one more time! This time, however, we'll make the request through the session we set up.

In [7]:
query = '''
    {
        items(full_identifier: "ABC") {
            total
            next_page
            results {
                full_identifier
                title
            }
        }
    }
'''
response = session.post(API_URL, json={'query': query})

response.json()

{'data': {'items': {'total': 1,
   'next_page': None,
   'results': [{'full_identifier': 'WD1-ABC',
     'title': 'Tum-why-village-moved-here'}]}}}

And there we go! We've successfully made a query to **PARADISEC**'s **API**, as the cookies associated with our session give us the authorization needed to make requests. Of course, we probably don't want to work with all the information our desired data was wrapped in, right? Well, `.json()` doesn't only get the JSON of the response, but turns it into a dictionary! This, combined with the fact that our output is hierarchical to match the structure of our query, makes it quite easy to retrieve just the data we want. The full response becomes a series of nested dictionaries, and we just use the names of the *field*s we queried on as the keys to access their values. So, let's print out each of those values by themselves:

In [8]:
print(response.json()['data']['items']['total'])
print(response.json()['data']['items']['next_page'])

for result in response.json()['data']['items']['results']:
    print(result['full_identifier'])
    print(result['title'])

1
None
WD1-ABC
Tum-why-village-moved-here


Once again, `'results'` looks a little different from the others when accessing the data, because it's a list. We would get an error if we tried to just do<br>`...['results']['full_identifier']`, because we need to access a specific member within the list returned by `'results'` first. As above, a simple solution is to just iterate through each item in the list, but of course depending on one's needs one can work with the list in any variety of ways. Also, as a reminder, the first key will always be `'data'` before we pass in the *field*s we actually queried on.

That's all there is to it! Soon, you'll start devising your own queries, but let's cover one more helpful **GraphQL** concept: ***variable*s**.

### Why Do We Need Variables?

So far, every *argument* value we've used has been hard-coded directly into the query string, like `"ABC"` in `items(full_identifier: "ABC")`. In a real application, though, we'll usually want that value to change - based on user input, a dropdown selection, a search box, and so on.

We *could* try to build a new query string every time by pasting the new value directly into the text. But the official [**GraphQL** documentation](https://graphql.org/learn/queries/#variables) specifically warns against this: it forces the client to manipulate query strings at runtime, and, in their words, "we should never be doing string interpolation to construct queries from user-supplied values." *Variable*s are **GraphQL**'s built-in, safer way to swap out just the values in a query, without touching the rest of its structure.

### Using a Variable: Three Steps

Per the **GraphQL** docs, using a *variable* always takes three steps:

1. **Declare** the *variable* at the start of the query, along with its *type*.
2. **Use** the *variable* in place of the literal value, wherever the *argument* appears.
3. **Supply** the actual value for the *variable* separately, alongside the query.

Let's walk through each step using our familiar `full_identifier` example.

**1. Declaring a variable**

*Variable*s are always named with a leading `$`, and can be called anything - for simplicity, we'll usually name ours the same as the *argument* they fill in. Recall that opening a query with `{` is really shorthand for `query {`; declaring *variable*s is where that shorthand runs out, because we need somewhere to list them. A query that declares a `full_identifier` *variable* starts like this:

`query($full_identifier: String) {`

This says: "this query accepts one *variable*, named `full_identifier`, which must be a `String`." Just like *field*s, *variable*s must be given a *type* - and here, that *type* is the `String` *Scalar* we already know.

**2. Using the variable**

Once declared, the *variable* replaces the literal value inside the *argument*'s parentheses:

`items(full_identifier: $full_identifier) {`

Instead of the fixed string `"ABC"`, `items` now receives whatever value is passed in for `$full_identifier`.

**3. Supplying a value**

By default, *variable*s are *nullable*, just like *argument*s - if no value is supplied, it's treated as though the *argument* was left out entirely. Two optional additions can change this:

- **Require a value** by adding `!` after the *type*: `query($full_identifier: String!) {`. The query will now return an error unless a real value is supplied.
- **Set a fallback** by adding `=` and a value after the *type*: `query($full_identifier: String = "ABC") {`. This value is used automatically whenever none is supplied.

Finally, to actually run the query, we still need to give the *variable* its value. In Python, that means three small steps: store the desired value in a regular Python variable, put it into a dictionary that matches each **GraphQL** *variable* name to its value, and pass that dictionary alongside our query string, under the key `'variables'`. Let's put it all together and run it below!

In [9]:
value = 'ABC'

query = '''
    query($full_identifier: String) {
        items(full_identifier: $full_identifier) {
            total
            next_page
            results {
                full_identifier
                title
            }
        }
    }
'''
variables = {'full_identifier': value}
response = session.post(API_URL, json={'query': query, 'variables': variables})

response.json()

{'data': {'items': {'total': 1,
   'next_page': None,
   'results': [{'full_identifier': 'WD1-ABC',
     'title': 'Tum-why-village-moved-here'}]}}}

Just like before, we got exactly the output we expected, even after our modifications! And now, if we wanted to make the same query, just with a different identifier, we only have to change the value of our variable and post the query again, rather than having to write an entirely new one.

In [10]:
value = 'AAA'
variables = {'full_identifier': value}
response = session.post(API_URL, json={'query': query, 'variables': variables})

response.json()

{'data': {'items': {'total': 1,
   'next_page': None,
   'results': [{'full_identifier': 'WD1-AAA', 'title': 'Geiba-to-Wagu'}]}}}

You may be wondering why we assign the value to a variable first rather than just define it in the dictionary; that's because if we're using *variable*s then we normally wouldn't be inputting values directly in the code, but retrieving them from some user, and we would generally expect to already have those values stored in a variable from elsewhere. In this notebook, we can use `input()` for that purpose to demonstrate what it would look like a bit more accurately - try inputting your own identifier below!

In [11]:
value = input('Assign value:')
variables = {'full_identifier': value}
response = session.post(API_URL, json={'query': query, 'variables': variables})

response.json()

Assign value: MMT1


{'data': {'items': {'total': 73,
   'next_page': 2,
   'results': [{'full_identifier': 'MMT1-20160711Bililuna',
     'title': 'Jack Gordon, Marie Gordon, Patrick  Smith & Marie Mudgell'},
    {'full_identifier': 'MMT1-20160716PO',
     'title': 'Interview with Patrick Jungarrayi Ooladoodi'},
    {'full_identifier': 'MMT1-20170703a',
     'title': 'Interview with Hairbrush George Jungarrayi'},
    {'full_identifier': 'MMT1-20170703b',
     'title': 'Interview with Patrick Ooladoodi Tjungarrayi & Charlie Tjapangardi Warlampirri'},
    {'full_identifier': 'MMT1-20171005TiTree',
     'title': 'Interview with Joe Bird and Albie Ampetyane'},
    {'full_identifier': 'MMT1-20171104KW',
     'title': 'Kathleen Kemarre Wallace'},
    {'full_identifier': 'MMT1-20171109',
     'title': 'Iluwanti Ken, Rene Kulitja, Josephine Mick & Tinpulya Mervyn'},
    {'full_identifier': 'MMT1-20171224NP', 'title': 'Nellie Patterson'},
    {'full_identifier': 'MMT1-20180501PH',
     'title': 'Bobby West, Alice S

### Querying a List of Terms

Sometimes we don't just want one value - we have a whole list of terms we're interested in, and want a set of responses back, one per term. You might expect to just pass that list in as a single *variable*, but if we check **PARADISEC**'s *schema*, every *argument* under `items` only accepts a single value (a `String`, `Int`, etc.), never a list. So we can't hand it a list directly.

Instead, we can reuse the same *variable*-based query as before, and simply loop over our list of terms in Python, making one request per term and collecting each response as we go:

In [34]:
terms = ['ABC', 'AAA', 'MMT1']

query = '''
    query($full_identifier: String) {
        items(full_identifier: $full_identifier) {
            total
            next_page
            results {
                full_identifier
                title
            }
        }
    }
'''

all_results = {}
for term in terms:
    variables = {'full_identifier': term}
    response = session.post(API_URL, json={'query': query, 'variables': variables})
    all_results[term] = response.json()

all_results

{'ABC': {'data': {'items': {'total': 1,
    'next_page': None,
    'results': [{'full_identifier': 'WD1-ABC',
      'title': 'Tum-why-village-moved-here'}]}}},
 'AAA': {'data': {'items': {'total': 1,
    'next_page': None,
    'results': [{'full_identifier': 'WD1-AAA', 'title': 'Geiba-to-Wagu'}]}}},
 'MMT1': {'data': {'items': {'total': 73,
    'next_page': 2,
    'results': [{'full_identifier': 'MMT1-20160711Bililuna',
      'title': 'Jack Gordon, Marie Gordon, Patrick  Smith & Marie Mudgell'},
     {'full_identifier': 'MMT1-20160716PO',
      'title': 'Interview with Patrick Jungarrayi Ooladoodi'},
     {'full_identifier': 'MMT1-20170703a',
      'title': 'Interview with Hairbrush George Jungarrayi'},
     {'full_identifier': 'MMT1-20170703b',
      'title': 'Interview with Patrick Ooladoodi Tjungarrayi & Charlie Tjapangardi Warlampirri'},
     {'full_identifier': 'MMT1-20171005TiTree',
      'title': 'Interview with Joe Bird and Albie Ampetyane'},
     {'full_identifier': 'MMT1-2017

### Putting It All Together

Let's combine everything we've covered so far: a query that pulls back a wide range of *field*s (not just `full_identifier` and `title`), including several *Object* and list *field*s, run against a real item. Rather than just printing the raw dictionary we get back, let's also format the output so it's actually legible.

In [38]:
query = '''
    query($full_identifier: String) {
        items(full_identifier: $full_identifier) {
            results {
                full_identifier
                title
                description
                region
                dialect
                originated_on
                doi
                access_condition_name
                collection {
                    title
                }
                collector {
                    name
                }
                university {
                    name
                }
                content_languages {
                    name
                }
                data_types {
                    name
                }
                essences {
                    filename
                    permalink
                }
            }
        }
    }
'''
variables = {'full_identifier': 'AA1-001'}
response = session.post(API_URL, json={'query': query, 'variables': variables})
results = response.json()['data']['items']['results']

for item in results:
    print(f"Title: {item['title']}")
    print(f"Identifier: {item['full_identifier']}")
    print(f"Description: {item['description']}")
    print(f"Region: {item['region']}")
    print(f"Dialect: {item['dialect']}")
    print(f"Originated On: {item['originated_on']}")
    print(f"DOI: {item['doi']}")
    print(f"Access Condition: {item['access_condition_name']}")
    print(f"Collection: {item['collection']['title']}")
    print(f"Collector: {item['collector']['name']}")
    print(f"University: {item['university']['name']}")
    print(f"Languages: {', '.join(lang['name'] for lang in item['content_languages'])}")
    print(f"Data Types: {', '.join(dt['name'] for dt in item['data_types'])}")
    print("Essences:")
    for essence in item['essences']:
        print(f"  - {essence['filename']}: {essence['permalink']}")
    print()

Title: Pak Nongeng tells about the first four people, food taboos, and various other topics
Identifier: AA1-001
Description: Pak Nongeng tells about the first four people, food taboos, and various other topics
Region: Sasak village, Samalantan, Sambas Regency, West Kalimantan
Dialect: None
Originated On: 1987-11-17
DOI: 10.4225/72/56E97A6DBA4F4
Access Condition: Open (subject to agreeing to PDSC access conditions)
Collection: Recordings of Selako (Indonesia)
Collector: Alexander Adelaar
University: University of Melbourne
Languages: Kendayan
Data Types: Sound
Essences:
  - AA1-001-A.mp3: https://catalog.paradisec.org.au/repository/AA1/001/AA1-001-A.mp3
  - AA1-001-A.wav: https://catalog.paradisec.org.au/repository/AA1/001/AA1-001-A.wav
  - AA1-001-B.mp3: https://catalog.paradisec.org.au/repository/AA1/001/AA1-001-B.mp3
  - AA1-001-B.wav: https://catalog.paradisec.org.au/repository/AA1/001/AA1-001-B.wav

Title: Pak Kaslem tells: 2 folk stories + accounts of customary law
Identifier: AA1

### Searching inside the `description` Argument

Let's put this same testing approach to work on another *field*. `items` also has a `description` *argument*, which looks like it should let us search for *item*s whose `description` contains a given word - for example, "music" or "song". But once again, the *schema* offers no explanation of how the match actually works: exact match, substring match, real full-text search? The only way to know is to test it, and check whether the results we get back actually contain the word we searched for.

In [43]:
query = '''
    query($description: String) {
        items(description: $description) {
            total
            results {
                full_identifier
                title
                description
            }
        }
    }
'''

for term in ['music', 'song']:
    variables = {'description': term}
    response = session.post(API_URL, json={'query': query, 'variables': variables})
    data = response.json()['data']['items']
    print(f"Search term: '{term}'")
    print(f"Total matching items: {data['total']}")
    for item in data['results']:
        contains_term = term.lower() in (item['description'] or '').lower()
        print(f"  {item['full_identifier']}: {item['title']} (description contains '{term}': {contains_term})")
    print()

Search term: 'music'
Total matching items: 883
  AC1-004: Banyata (description contains 'music': True)
  AC1-007: Sikayana (description contains 'music': True)
  AC1-009: Kohaivi (description contains 'music': True)
  AC1-011: Loun; Gao; Maringe; Kia (description contains 'music': True)
  AC1-017: Recorded from Radio (Lebanese Arabic) (description contains 'music': True)
  AC1-111: Bauro, Malango, Gaobata, Gengo, Dai, Banyata (Gospel recording copy) (description contains 'music': True)
  AC1-126: Miao panpipers, singing, kinship. (description contains 'music': True)
  AC1-425: Languages of the USSR (description contains 'music': True)
  ACLA1-FM059_B: FM059_B (description contains 'music': True)
  ACLA1-FM059_B_video: FM059_B_video (description contains 'music': True)

Search term: 'song'
Total matching items: 2172
  AA2-002: Baranangis  + The history of Baki' Reang (by Piang Teba') (description contains 'song': True)
  AAZ2019-CON020: a song (description contains 'song': True)
  AAZ20

***
## Do It Yourself

Now you should be armed with all the knowledge you need to start testing our own queries and explore **PARADISEC**! Again, the *schema* definition for their specific **GraphQL** architecture can be found [here](https://admin-catalog.paradisec.org.au/paradisec.graphql). Every query you make must begin at one of the *field*s contained within the `Query` *type*. Feel free to try using *variable*s or supply all arguments directly within the code as you see fit. The original example query we've been using will be provided below as a framework to alter and build off of, as well as a few empty cells after that - also feel free to add more cells! Once you're satisfied with your testing, move on to the next section.

In [44]:
query = '''
    {
        items(full_identifier: "ABC") {
            total
            next_page
            results {
                full_identifier
                title
            }
        }
    }
'''
response = session.post(API_URL, json={'query': query})

response.json()

{'data': {'items': {'total': 1,
   'next_page': None,
   'results': [{'full_identifier': 'WD1-ABC',
     'title': 'Tum-why-village-moved-here'}]}}}

***
## API Problems

Now that you've explored the **API** on your own some more, you may have noticed some discrepancies or recieved unexpected returns. For an example of what I mean, let's try to get all the items that originated from Yale University.

In [36]:
query = '''
    {
        items(university_name: "Yale University") {
            total
            next_page
            results {
                university {
                    name
                }
            }
        }
    }
'''
response = session.post(API_URL, json={'query': query})

response.json()

{'data': {'items': {'total': 39181,
   'next_page': 2,
   'results': [{'university': {'name': 'Australian National University'}},
    {'university': {'name': 'Australian National University'}},
    {'university': {'name': 'Australian National University'}},
    {'university': {'name': 'Australian National University'}},
    {'university': {'name': 'Australian National University'}},
    {'university': {'name': 'Australian National University'}},
    {'university': {'name': 'Australian National University'}},
    {'university': {'name': 'Australian National University'}},
    {'university': {'name': 'Australian National University'}},
    {'university': {'name': 'Australian National University'}}]}}}

Well...that didn't seem to work at all. And yet, it did still return a response, so it's not as though there's something wrong with the query - it simply didn't do what we expected it to do at all. In fact, if we test running this with no arguments at all:

In [37]:
query = '''
    {
        items {
            total
        }
    }
'''
response = session.post(API_URL, json={'query': query})

response.json()

{'data': {'items': {'total': 39181}}}

We get the exact same total amount of items! In other words, supplying an argument to `university_name` did effectively nothing at all. This is an example of another issue one may run into when working with an **API**: nothing is guaranteed to work. If an **API** is accessible, then we can assume that it's been finished and everything has been adequately tested, but there are rare occasionas where this isn't the case. What's also more likely is that it works perfectly at one point, then is updated in a way that accidentally breaks something and the new bug isn't caught. Whatever the reason, sometimes code just doesn't work right. While frustrating, there's generally not much you can do in this case. The only course of action that could be generally recommended is to contact the developers of the **API** and make sure they're aware of any problems.

Now, throughout all of this, there's one underlying thread that I haven't yet touched on. Thinking back to a variety of things mentioned up to this point - we never gave the other answer to our grouped questions a while back, about how we would know what passing in arguments would do. How did we find **PARADISEC**'s *schema* definition and their *endpoint*? How could we be sure that all of our setup for our queries was correct? How did we know to implement our more complicated login process? Most importantly of all, how are we supposed to know what any of the queries available to us <u>actually</u> do? The comments in the *schema* aren't always in depth, so while we've been getting by with the fact that **GraphQL** is supposed to be somewhat intuitively understandable with the naming of queries and *type*s, is assumptions really all we have to go off of? Especially, again, when trying to determine what a *field* will actually do with any of the *argument*s passed into it.

All of these questions come back to one broad answer which was brushed over once earlier: [*documentation*](https://blog.hubspot.com/website/api-documentation). **API** *documentation* is a collection of all the information one working with an **API** could want to have. Good *documentation* will have all kinds of information: what the *endpoint*s are, how to properly authenticate requests, explanations and code examples of every method and every argument, a changelog, if there are rate limits (how many calls you can make to the **API** within some amount of time), terms of use, a link to support, etc. Good *documentation* will have the answers to all of our questions above, and plenty of extra information too. **API** *documentation* is an essential resource when working with any **API**, regardless of scope or experience. So why didn't we show **PARADISEC**'s **API** *documentation* earlier?

As we mentioned in our first description of possible **API** issues, *documentation* can sometimes be outdated or inaccurate. As hinted at by the addition of the word "good" when describing *documentation* above, there is also a variety of ways in which *documentation* can be poorly written: difficult to navigate or search for specific methods, a lack of detail in explanations/examples, or overly technical writing that makes it exceedingly difficult to understand anything if you don't already have a good understanding of how the **API** works. These are a handful of the much more minor problems that can arise when working with **API**s, but a significantly worse problem? A lack of *documentation* entirely. **PARADISEC** has a small portion of one section of their website that contains their *endpoint* (which is actually slightly wrong), the example query we've been using, and their *schema* defintion. <U>That is the extent of their **GraphQL** *documentation*</u>. We've already laid out all the information we were given.

To be clear, this isn't meant to decry **PARADISEC**; it's an archive focused on preserving language and culture, so developing and maintaining their **API** simply isn't a top priority, and understandably so. Instead, this highlights two important points. The first is that **API**s can sometimes be exceedingly difficult to work with. In worst-case scenarios, you can end up with no information and no support. While not particularly common, an **API** without any documentation is far from unheard of. The second point is that even under these circumstances, there's still a way forward. The login process we worked through above was not provided by **PARADISEC** - it was developed through trying to understand why communicating with the **API** wasn't working and trial-and-error. Even when no information is given about a particular part of an **API** or the **API** as a whole, one can gather that information for themself through continuous testing.

 ***
## Conclusion

While different **API**s can be quite unique, the basics of communicating with one is often quite similar. Identify the desired *endpoint*(s), check what kind of *authentication* process is needed, then start making requests! All of the **API**-specific information is contained in the **API** *documentation*, an invaluable resource that should always be consulted first when working with any **API**.

No **API** is perfect. Some may have faulty or poor *documentation*; some may not work as they should; some may be lacking information entirely. However, problems are rarely insurmountable. Proper debugging and testing can often help one to achieve their goals. The kind of **API** one is working with may also influence how many problems they face - an **API** that is widely used or maintained by a large company will probably have less problems, while an **API** that is particularly niche or supported only by a small team may be more difficult to work with.

**GraphQL** is an **API** architecture focused on efficient data retrieval. It is structured around *type*s and *field*s, and is queried on through one *endpoint*. One query can specify exactly what information is being requested, which prevents repeated requests, over-fetching, and under-fetching.

**PARADISEC** is a digital archive dedicated to preserving many of the small languages and cultures of the world. Now that you've explored much of their metadata, go [explore](https://catalog.paradisec.org.au/search) what's actually been preserved!